In [1]:
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

RUTA_PROYECTO = Path("..")
RUTA_PROCESSED = RUTA_PROYECTO / "data" / "procesado"

carpeta_etiquetadas = (
    RUTA_PROCESSED / "resenas_etiquetadas"
)

archivos_etiquetados = sorted(
    carpeta_etiquetadas.glob(
        "resenas_etiquetadas_*.parquet"
    )
)

print(
    "Archivos etiquetados encontrados:",
    len(archivos_etiquetados)
)

Archivos etiquetados encontrados: 47


In [4]:
# Columnas necesarias

columnas_modelado = [
    "review_text_processed",    #Variable entrada
    "sentiment",    #Variable objetivo
    "parent_asin"   #Para saber qué videojuego se está tratando
]

bloque_modelado = []

for archivo in archivos_etiquetados:
    bloque = pd.read_parquet(
        archivo,
        columns=columnas_modelado
    )
    
    bloque_modelado.append(bloque)

In [6]:
datos_modelado = pd.concat(bloque_modelado, ignore_index=True)

print(
    "Reseñas cargadas:",
    f"{len(datos_modelado):,}"
)

print(
    "Textos vacíos:",
    datos_modelado["review_text_processed"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Etiquetas nulas:",
    datos_modelado["sentiment"]
    .isna()
    .sum()
)

Reseñas cargadas: 1,485,343
Textos vacíos: 0
Etiquetas nulas: 0


In [8]:
    #Revisar distribución
distribucion_sentimientos = (
    datos_modelado["sentiment"]
    .value_counts()
)

porcentajes_sentimientos = (
    datos_modelado["sentiment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .astype(str) + "%"
)

resumen_sentimientos = pd.DataFrame({
    "Cantidad": distribucion_sentimientos,
    "Porcentaje": porcentajes_sentimientos
})

resumen_sentimientos

,Cantidad,Porcentaje
sentiment,,
positivo,1129067,76.01%
negativo,238483,16.06%
neutral,117793,7.93%


Respecto a las variables:
- X: variables de entrada, el texto procesado de cada reseña
- Y: variables objetivo, el sentimiento que tiene que predecir
- train_test_split divide x e y en entrenamiento y prueba: X_test X_train y_test y_train
- X_train: textos para entrenar al modelo
- y_train: etiquetas de sentimiento correspondientes
- X_test: texto para probar el entrenamiento
- y_test: etiquetas correspondientes

In [14]:
#Definir variables

X = datos_modelado["review_text_processed"]
y = datos_modelado["sentiment"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [15]:
print("DIVISIÓN DEL CONJUNTO")
print("---------------------")
print(
    "Reseñas de entrenamiento:",
    f"{len(X_train):,}"
)
print(
    "Reseñas de prueba:",
    f"{len(X_test):,}"
)
print(
    "Total:",
    f"{len(X_train) + len(X_test):,}"
)

DIVISIÓN DEL CONJUNTO
---------------------
Reseñas de entrenamiento: 1,188,274
Reseñas de prueba: 297,069
Total: 1,485,343


In [12]:
print("DISTRIBUCIÓN EN ENTRENAMIENTO")
print("-----------------------------")
print(
    y_train
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nDISTRIBUCIÓN EN PRUEBA")
print("----------------------")
print(
    y_test
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

DISTRIBUCIÓN EN ENTRENAMIENTO
-----------------------------
sentiment
positivo    76.01
negativo    16.06
neutral      7.93
Name: proportion, dtype: float64

DISTRIBUCIÓN EN PRUEBA
----------------------
sentiment
positivo    76.01
negativo    16.06
neutral      7.93
Name: proportion, dtype: float64


In [16]:
print("\nCOMPROBACIÓN")
print("------------")
print(
    "¿Se conservan todas las reseñas?",
    len(X_train) + len(X_test) == len(datos_modelado)
)

print(
    "¿Coinciden X_train e y_train?",
    len(X_train) == len(y_train)
)

print(
    "¿Coinciden X_test e y_test?",
    len(X_test) == len(y_test)
)


COMPROBACIÓN
------------
¿Se conservan todas las reseñas? True
¿Coinciden X_train e y_train? True
¿Coinciden X_test e y_test? True


In [18]:
# vectorizador

import numpy as np


vectorizador_tfidf = TfidfVectorizer(
    max_features=50_000, # Limita a 50000 términos más relevantes
    min_df=5,   # El término tiene que aparecer mínimo 5 veces
    max_df=0.95,    # Como máximo un término puede aparecer hasta en un 95% de las reseñas
    ngram_range=(1, 2), # análisis para un término y pareja
    sublinear_tf=True,  # frecuencia suavizada
    dtype=np.float32
)

Para probarlo en una muestra se tiene que hacer un vectorizador prueba, sino este aprendería con los datos de prueba. También se podría eliminar la matriz resultante y volver a usar el mimso vectorizador

In [20]:
# Probar en muestra
cantidad_prueba = min(
    100_000,
    len(X_train)
)

X_train_prueba = X_train.iloc[
    :cantidad_prueba
]

print(
    "Textos utilizados en la prueba:",
    f"{len(X_train_prueba):,}"
)

Textos utilizados en la prueba: 100,000


In [21]:
vectorizador_prueba = TfidfVectorizer(
    max_features=50_000, # Limita a 50000 términos más relevantes
    min_df=5,   # El término tiene que aparecer mínimo 5 veces
    max_df=0.95,    # Como máximo un término puede aparecer hasta en un 95% de las reseñas
    ngram_range=(1, 2), # análisis para un término y pareja
    sublinear_tf=True,  # frecuencia suavizada
    dtype=np.float32
)

X_train_tfidf_prueba = (
    vectorizador_prueba.fit_transform(
    X_train_prueba
    )
)

In [23]:
print("PRUEBA DE VECTORIZACIÓN")
print("-----------------------")
print(
    "Dimensiones de la matriz:",
    X_train_tfidf_prueba.shape
)
print(
    "Términos del vocabulario:",
    len(vectorizador_prueba.vocabulary_)
)
print(
    "Tipo de matriz:",
    type(X_train_tfidf_prueba)
)
print(
    "Tipo de los valores:",
    X_train_tfidf_prueba.dtype
)

PRUEBA DE VECTORIZACIÓN
-----------------------
Dimensiones de la matriz: (100000, 50000)
Términos del vocabulario: 50000
Tipo de matriz: <class 'scipy.sparse._csr.csr_matrix'>
Tipo de los valores: float32


En la matriz cada fila representa una reseña y cada columna una característica del vocabulario

In [24]:
# Revisar características

caracteristicas = (
    vectorizador_prueba
    .get_feature_names_out()
)

print(
    "Primeras características:",
    caracteristicas[:20]
)

print(
    "\nÚltimas características:",
    caracteristicas[-20:]
)

Primeras características: ['00' '00 game' '00 not' '000' '000 000' '000 point' '000 year' '007' '01'
 '02' '03' '04' '05' '06' '07' '08' '09' '10' '10 00' '10 000']

Últimas características: ['zombie game' 'zombie good' 'zombie kill' 'zombie like' 'zombie map'
 'zombie mode' 'zombie not' 'zombie zombie' 'zone' 'zone game' 'zoning'
 'zoo' 'zoo tycoon' 'zoom' 'zork' 'zuma' 'zumba' 'zumba class'
 'zumba fitness' 'zumba wii']


In [ ]:
# Comprobr que hay algunos términos importantes
terminos_interes = [
    "not good",
    "not recommend",
    "highly recommend",
    "great game",
    "waste money"
]

for termino in terminos_interes:
    print(
        termino,
        ":",
        termino in vectorizador_prueba.vocabulary_
    )

not good : True
not recommend : True
highly recommend : True
great game : True
waste money : True


In [26]:
# Características aleatorias

import numpy as np

np.random.seed(42)

caracteristicas_aleatorias = np.random.choice(
    caracteristicas,
    size=50,
    replace=False
)
print(caracteristicas_aleatorias)

['persona series' 'decent' '16 hour' 'event add' 'search engine'
 'story set' 'download try' 'worthy successor' 'blackmoor' 'raider series'
 'sucker' 'rounded' 'character go' 'addict not' 'gift play'
 'nearly perfect' 'friend list' 'play keyboard' 'excellent sequel'
 'entail' 'character bring' 'game advertise' '95 time' 'freeze time'
 'fantastic experience' 'love campaign' 'star story' 'no difference'
 'real guitar' 'pat' 'purchase enjoy' 'later play' 'car handle'
 'big challenge' 'right length' 'heartfelt' 'yeah good' 'starve'
 'want challenge' 'dumber' 'soundtrack cd' 'iam' 'debris' 'reviewer give'
 'no local' 'somewhat fun' 'flak' 'lot gun' 'minute long' 'game treat']


In [28]:
# más frecuentes

frecuencia_caracteristicas = np.asarray(
    X_train_tfidf_prueba.sum(axis=0)
).ravel()

indices_mayores = frecuencia_caracteristicas.argsort()[
    -30:
][::-1]

terminos_mayor_peso = [
    (
        caracteristicas[indice],
        frecuencia_caracteristicas[indice]
    )
    for indice in indices_mayores
]

for termino, peso in terminos_mayor_peso:
    print(
        f"{termino:<30} {peso:.2f}"
    )

game                           4824.57
star                           4529.37
great                          3027.30
not                            2769.54
good                           2638.58
love                           2551.36
play                           2269.87
fun                            2095.82
great game                     1616.40
like                           1596.70
buy                            1302.38
star great                     1298.06
work                           1236.30
time                           1220.55
star good                      1079.97
good game                      1067.00
love game                      1008.37
get                            999.85
awesome                        985.99
no                             904.41
son                            868.59
graphic                        815.21
come                           805.50
new                            792.95
kid                            785.71
fun game                       78

In [29]:
# solo números

caracteristicas_numericas = [
    termino
    for termino in caracteristicas
    if termino.replace(" ", "").isdigit()
]

print(
    "Características formadas solo por números:",
    len(caracteristicas_numericas)
)

print(
    "Porcentaje del vocabulario:",
    f"{len(caracteristicas_numericas) / len(caracteristicas) * 100:.2f} %"
)

print(
    "Ejemplos:",
    caracteristicas_numericas[:30]
)

Características formadas solo por números: 265
Porcentaje del vocabulario: 0.53 %
Ejemplos: ['00', '000', '000 000', '007', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '10 00', '10 000', '10 10', '10 11', '10 12', '10 13', '10 15', '10 20', '10 30', '100', '100 000', '100 100', '1000', '101', '1024', '108']


In [30]:
# eliminar variables temporales

del X_train_tfidf_prueba
del X_train_prueba

In [31]:
# Ajustar vectorizador inicial

X_train_tfidf = (
    vectorizador_tfidf.fit_transform(
        X_train
    )
)

In [32]:
X_test_tfidf = (
    vectorizador_tfidf.transform(
        X_test
    )
)

In [33]:
print("RESULTADO DE LA VECTORIZACIÓN")
print("-----------------------------")

print(
    "Dimensiones de entrenamiento:",
    X_train_tfidf.shape
)

print(
    "Dimensiones de prueba:",
    X_test_tfidf.shape
)

print(
    "Tamaño del vocabulario:",
    len(vectorizador_tfidf.vocabulary_)
)

print(
    "Tipo de matriz de entrenamiento:",
    type(X_train_tfidf)
)

print(
    "Tipo de los valores:",
    X_train_tfidf.dtype
)

RESULTADO DE LA VECTORIZACIÓN
-----------------------------
Dimensiones de entrenamiento: (1188274, 50000)
Dimensiones de prueba: (297069, 50000)
Tamaño del vocabulario: 50000
Tipo de matriz de entrenamiento: <class 'scipy.sparse._csr.csr_matrix'>
Tipo de los valores: float32


In [34]:
print("\nCOMPROBACIONES")
print("--------------")

print(
    "¿Coinciden las filas de entrenamiento?",
    X_train_tfidf.shape[0] == len(y_train)
)

print(
    "¿Coinciden las filas de prueba?",
    X_test_tfidf.shape[0] == len(y_test)
)

print(
    "¿Ambas matrices tienen las mismas características?",
    X_train_tfidf.shape[1] == X_test_tfidf.shape[1]
)


COMPROBACIONES
--------------
¿Coinciden las filas de entrenamiento? True
¿Coinciden las filas de prueba? True
¿Ambas matrices tienen las mismas características? True
